In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
!nvidia-smi

Tue Jan  6 01:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 1. Load Environment

In [5]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [6]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)

transformers: 4.57.3
datasets: 4.0.0
accelerate: 1.12.0
peft: 0.18.0
bitsandbytes: 0.49.0
trl: 0.26.2


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Import Libraries

In [8]:
import math
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# 3. Load Dataset

In [9]:
name_dataset = 'SSTv2'

In [10]:
dataset = load_dataset("glue", "sst2")

In [11]:
dataset = dataset.rename_column("sentence", "text")

In [12]:
dataset = dataset.select_columns(["text", "label"])

In [13]:
dataset.shape

{'train': (67349, 2), 'validation': (872, 2), 'test': (1821, 2)}

In [14]:
dataset = dataset['train']

In [15]:
split_1 = dataset.train_test_split(test_size=0.3, stratify_by_column = 'label', seed=42)

In [16]:
dataset_train = split_1['train']
split_test = split_1['test']

In [17]:
split_2 = split_test.train_test_split(test_size=0.5, stratify_by_column = 'label', seed=42)

In [18]:
dataset_val = split_2['train']
dataset_test = split_2['test']

In [19]:
df_train = dataset_train.to_pandas()
df_val = dataset_val.to_pandas()
df_test = dataset_test.to_pandas()

**a. Analysis: Train Set**

In [20]:
df_train.shape

(47144, 2)

In [21]:
df_train['label'].value_counts()

,count
label,
1,26298
0,20846


In [22]:
round(df_train['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
1,55.78
0,44.22


**b. Analysis: Validation Set**

In [23]:
df_val.shape

(10102, 2)

In [24]:
df_val['label'].value_counts()

,count
label,
1,5635
0,4467


In [25]:
round(df_val['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
1,55.78
0,44.22


**c. Analysis: Test Set**

In [26]:
df_test.shape

(10103, 2)

In [27]:
df_test['label'].value_counts()

,count
label,
1,5636
0,4467


In [28]:
round(df_test['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
1,55.79
0,44.21


**d. Save dataframes**

In [29]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/01.Datasets_Creation/{name_dataset}'

In [30]:
df_train.to_csv(f'{path_save}/df_train.csv')
df_val.to_csv(f'{path_save}/df_val.csv')
df_test.to_csv(f'{path_save}/df_test.csv')

In [31]:
N = 5
chunk_size = math.ceil(len(df_test) / N)
splits_test = [
    df_test.iloc[i:i + chunk_size]
    for i in range(0, len(df_test), chunk_size)
]

In [32]:
print(len(splits_test))
print(splits_test[0].shape)

5
(2021, 2)


In [33]:
for i in range(N):
  idx = i + 1
  splits_test[i].to_csv(f'{path_save}/df_test_{idx}.csv')

# 4. BERT

In [34]:
name_model = "bert-base-uncased"

In [35]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [36]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [37]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [38]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/10102 [00:00<?, ? examples/s]

In [39]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

In [40]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/47144 [00:00<?, ? examples/s]

In [41]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/10102 [00:00<?, ? examples/s]

In [42]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/10103 [00:00<?, ? examples/s]

# 5. DistilBERT

In [43]:
name_model = "distilbert-base-uncased"

In [44]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [45]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [46]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [47]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/10102 [00:00<?, ? examples/s]

In [48]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

In [49]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/47144 [00:00<?, ? examples/s]

In [50]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/10102 [00:00<?, ? examples/s]

In [51]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/10103 [00:00<?, ? examples/s]

# 6. RoBERTa

In [52]:
name_model = "roberta-base"

In [53]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [54]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [55]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [56]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/10102 [00:00<?, ? examples/s]

In [57]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

In [58]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/47144 [00:00<?, ? examples/s]

In [59]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/10102 [00:00<?, ? examples/s]

In [60]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/10103 [00:00<?, ? examples/s]

# 7. Execution Time

In [61]:
end_notebook = time.time()

In [62]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 0m 31.79s
